In [3]:
#instalacion de la librería de DuckDB
!pip install duckdb

In [4]:
import duckdb

# Conectarnos a DuckDB en memoria
con = duckdb.connect(database=':memory:')

# Cargamos un dataset de clima real compatible
con.execute("""
    CREATE TABLE clima AS
    SELECT * FROM read_csv_auto('https://raw.githubusercontent.com/vega/vega-datasets/master/data/seattle-weather.csv')
""")

In [5]:
print("Columnas de nuestro dataset:")
display(con.execute("DESCRIBE clima;").fetchdf())

Columnas de nuestro dataset:


,column_name,column_type,null,key,default,extra
0,date,DATE,YES,None,None,None
1,precipitation,DOUBLE,YES,None,None,None
2,temp_max,DOUBLE,YES,None,None,None
3,temp_min,DOUBLE,YES,None,None,None
4,wind,DOUBLE,YES,None,None,None
5,weather,VARCHAR,YES,None,None,None


In [6]:
# 1. Limpieza básica (por ejemplo, filtrar filas donde la temperatura máxima no sea nula)
con.execute("""
    CREATE TABLE clima_limpio AS
    SELECT * FROM clima WHERE temp_max IS NOT NULL
""")

# 2. Consulta analítica 1: Promedios mensuales de temperatura máxima
print("--- Promedios mensuales de temperatura máxima ---")
promedios = con.execute("""
    SELECT EXTRACT(MONTH FROM date) AS mes, AVG(temp_max) AS temp_max_promedio
    FROM clima_limpio
    GROUP BY mes
    ORDER BY mes;
""").fetchdf()
display(promedios)

# 3. Consulta analítica 2: Máximos históricos de temperatura
print("--- Máximo histórico general ---")
maximos = con.execute("""
    SELECT MAX(temp_max) AS maxima_historica, MIN(temp_min) AS minima_historica
    FROM clima_limpio;
""").fetchdf()
display(maximos)

# 4. Consulta analítica 3: Filtrado complejo (días con más de 30 grados)
print("--- Días más calurosos (temp_max > 30) ---")
filtrados = con.execute("""
    SELECT date, temp_max, weather
    FROM clima_limpio
    WHERE temp_max > 30
    ORDER BY temp_max DESC;
""").fetchdf()
display(filtrados)

--- Promedios mensuales de temperatura máxima ---


,mes,temp_max_promedio
0,1,8.229032
1,2,9.860177
2,3,12.387097
3,4,15.020000
4,5,19.295968
5,6,22.400000
6,7,25.998387
7,8,26.112097
8,9,21.924167
9,10,16.389516


--- Máximo histórico general ---


,maxima_historica,minima_historica
0,35.6,-7.1


--- Días más calurosos (temp_max > 30) ---


,date,temp_max,weather
0,2014-08-11,35.6,rain
1,2015-07-19,35.0,sun
2,2012-08-16,34.4,sun
3,2014-07-01,34.4,sun
4,2015-07-30,34.4,sun
5,2015-07-31,34.4,sun
6,2012-08-04,33.9,sun
7,2012-08-05,33.9,sun
8,2013-06-30,33.9,sun
9,2013-09-11,33.9,sun
